# Lab 1 · Read One Digit

*From a list of 784 numbers to a machine that reads handwriting, with Python you wrote yourself.*

**Your mission:** Teach Python to read one handwritten digit, with functions you write yourself, and measure how often it gets it right.

**Reaching Putting It All Together (the end of the minimum path) is a complete lab. Everything after it is extra.**

**In this notebook**

1. Setup: run the first code cell
2. How this lab works: the markers ✏️ WRITE CODE and ▶️ JUST RUN, and the three parts CORE, OPTIONAL and ADVANCED
3. What is MNIST? The data, the labels, training and test images
4. Just enough theory
5. Fast lane: the whole CORE on one table
6. Load the data, Getting to Know Our Data, and the worked example W
7. Warm-up: predict the output (while others finish setup)
8. CORE: C1 → C2 → C3 → Putting It All Together → C5
9. After the CORE, with the whole class: the AI pit stop and Draw your own digit
10. OPTIONAL (extra practice): C4, ENG1, SCI1, BLD1
11. ADVANCED (for students who move faster): ENG2, SCI2, BLD2, the Pure-Python League
12. AI-use declaration, Before you go, What's next

Data: MNIST (LeCun, Cortes, Burges), CC BY-SA 3.0, adapted. Garabato Parcels is fictional.

In [ ]:
# ▶️ JUST RUN · Run me first (Shift+Enter). You don't need to read this cell.
LAB = 1
KIT_URL = ""   # filled in at build time if the kit is hosted; used only on Colab
import os, sys, platform
from pathlib import Path

def _find_kit(start):
    """Return the folder containing labkit/: here, up to 3 levels above, or 1 level below."""
    start = Path(start).resolve()
    for folder in [start] + list(start.parents)[:3]:
        if (folder / "labkit" / "__init__.py").is_file():
            return folder
    for child in sorted(p for p in start.iterdir() if p.is_dir()):
        if (child / "labkit" / "__init__.py").is_file():
            return child
    return None

print("Python", platform.python_version(), "|", sys.executable)
if sys.version_info < (3, 10):
    raise SystemExit("STOP: this kernel runs Python " + platform.python_version()
                     + ". The lab needs Python 3.10 or newer.\nFix: click the kernel name (top right) > "
                       "Select Another Kernel > Python Environments > pick 3.10+ (e.g. your course .venv). "
                       "START_HERE.md, Route A step 3, shows exactly where to click.")
if ".zip" in str(Path.cwd()).lower():
    raise SystemExit("STOP: the notebook was opened from inside the zip file.\nFix: close it, right-click "
                     "the zip > Extract All, then in VS Code use File > Open Folder on the extracted folder.")
IN_COLAB = "google.colab" in sys.modules
kit = _find_kit(Path.cwd())
if kit is None and IN_COLAB:
    import zipfile, urllib.request
    zips = sorted(Path("/content").glob("lab*_kit*.zip"))
    if not zips and KIT_URL:
        print("Colab: downloading the kit ...")
        urllib.request.urlretrieve(KIT_URL, "/content/kit.zip")
        zips = [Path("/content/kit.zip")]
    for z in zips:
        zipfile.ZipFile(z).extractall("/content")
    kit = _find_kit("/content")
if kit is None:
    raise SystemExit("STOP: I can't find the labkit folder.\nFix: open the WHOLE extracted kit folder in "
                     "VS Code (File > Open Folder), not only the notebook. On Colab, upload the kit zip "
                     "with the Files panel and run this cell again.")
os.chdir(kit)
if str(kit) not in sys.path:
    sys.path.insert(0, str(kit))
try:
    get_ipython().run_line_magic("load_ext", "autoreload")   # picks up edits in .py files
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    pass
import labkit
labkit.start(LAB)
from labkit import check, check_all, rescue, load_digits, show_digit, describe, evaluate, show_mistakes, try_predict, sketch_to_image, prepare_drawing, show_numbers, show_binary, show_greyscale, show_top3, show_rendered

## How this lab works

**Two markers tell you what to do with every cell:**

| Marker | What it means |
|---|---|
| **✏️ WRITE CODE** | You write or change code here. Stubs start with `# ===== YOUR CODE HERE =====`: replace `pass` with your code. |
| **▶️ JUST RUN** | Provided code: run it (Shift+Enter) and read what it prints. You don't need to change it. |

Helper functions in `labkit` are already implemented: use them, don't modify them.

**Three parts:**

| Part | Who and when | Feedback |
|---|---|---|
| **CORE** | everyone, in order: C1 → C2 → C3 → Putting It All Together (the minimum path: reaching it is a complete lab), then C5 | `check()`, 4 hints, `rescue()` |
| **OPTIONAL** | extra practice when your CORE is done, any task, any order: C4, ENG1, SCI1, BLD1 | `check()` and hints |
| **ADVANCED** | for students who move faster; open-ended, you can continue at home: ENG2, SCI2, BLD2, the League | a sanity check, a score, or your own measurements |

**Three tools.**
- `check("C1", squared_distance)` answers PASS, NOT YET or NOT STARTED. It shows the tests it ran (TEST 1, TEST 2, …),
  with the inputs, the expected result and your result; on NOT YET, the first failing test and a likely cause.
- **Hints** sit under each task. Open them one at a time.
- `squared_distance = rescue("C1")` hands you a working version, so the rest of the notebook keeps working.

**Stuck for 7 minutes after Hint 3? Use `rescue()`**, carry on, and come back to the task later.

**The AI dial** (a recommendation, not a ban):
- CORE: AI as **tutor**. Ask *why*, not for the answer; Hint 4 gives you a prompt. Try on your own for 7–10 minutes first.
- OPTIONAL and ADVANCED tasks: AI as **reviewer**. Keep only what you can explain line by line.
- The Challenge (ADVANCED): AI as **pair programmer**. Verify with checks and tests, and declare it at the end.
- Always: you own every line you run, you verify it, and you **never paste personal data into an AI tool**.

**Help card:** show **HELP PLEASE** when you'd like Javier to come over, **WORKING** otherwise.
Work alone or with a neighbour, your choice. Nothing here is graded or collected.

## What is MNIST?

**MNIST** is a famous collection of handwritten digits. It was put together in the 1990s by Yann LeCun,
Corinna Cortes and Christopher Burges, from digits written by American Census Bureau staff and high-school
students, and it has been used ever since to test programs that read handwriting.

- Each image shows **one digit, from 0 to 9**.
- Each image is **28 × 28 pixels**: 28 rows of 28 small squares.
- Python stores an image as **one list of 784 numbers** (28 × 28 = 784), from 0 (paper) to 255 (ink).
- Each image has a **label**: the digit it shows. People labelled every image beforehand, so we know the
  true answer for each one: the **ground truth**.

**Training images and test images.** We use the images in two groups:

- **Training images** are our references, the digits the program "knows". To read a new image, we look for
  the most similar training image: its label tells us the digit.
- **Test images** are new to the program. We know their labels too, but we use them **only to check** the
  program's answers, never to find them.

**The key idea:** we have images whose digit we already know. We use some as references to recognise
others, and then check whether we were right. (No neural networks today: just lists, loops and
comparisons.)

We'll work with 1,000 examples of handwritten digits and another 100 images to test our program.

**The plan of this lab, in six sentences**

| | Where |
|---|---|
| I have an image made of numbers. | *Getting to Know Our Data* |
| I know which digit each example is. | the labels |
| I can compare two images. | C1 |
| I can find the most similar one. | C2 |
| I can check whether I was right. | C3 and Putting It All Together |
| And now I can try to recognise my own digits. | *Draw your own digit* |

## Just enough theory

Open a primer only if the idea feels new. Each one is a short example you can run in your head.

<details>
<summary><strong>Lists and indexing (Theory S3)</strong></summary>

An image is one list of numbers. Positions start at 0.

```python
image = [0, 255, 128, 64]
image[0], image[-1], len(image)    # 0, 64 (the last one), 4
# A 28 x 28 image is one list of 784 numbers: pixel (r, c) is image[r * 28 + c]
```

</details>

<details>
<summary><strong>A for loop with a running total (Theory S2)</strong></summary>

Start a total before the loop, update it inside, use it after.

```python
total = 0
for x in [3, 1, 4]:
    total += x            # total is 3, then 4, then 8
print(total)              # 8; with positions: for i in range(len(a)): ... a[i] ...
```

</details>

<details>
<summary><strong>Counting with a dict (Theory S3)</strong></summary>

One key per thing you count; the value is how many so far. `.get(key, 0)` gives 0 for a new key.

```python
counts = {}
for label in [7, 1, 7]:
    counts[label] = counts.get(label, 0) + 1
print(counts)             # {7: 2, 1: 1}
```

</details>

<details>
<summary><strong>Functions, docstrings and type hints (Theory S1)</strong></summary>

`x: int` and `-> int` are type hints: notes for humans (Python doesn't enforce them). The docstring says
what the function does. `return` hands the value back to the caller; `print` only shows it.

```python
def double(x: int) -> int:
    """Return twice x."""
    return 2 * x
```

</details>

<details>
<summary><strong>f-strings (Theory S1)</strong></summary>

Put values inside `{}` in a string that starts with `f`. `:.1%` shows a fraction as a percentage.

```python
correct, total = 7, 8
print(f"{correct} of {total} correct ({correct / total:.1%})")   # 7 of 8 correct (87.5%)
```

</details>

## Fast lane

Some people like to read a spec and build; others like guided steps. Both routes reach the same place.

| Task | Signature | Spec |
|---|---|---|
| C1 (minimum path) | `squared_distance(a: list[int], b: list[int]) -> int` | the sum of (a[i] - b[i]) ** 2 over every position; the images have equal lengths |
| C2 (minimum path) | `predict_nearest(image: list[int], train_images: list[list[int]], train_labels: list[int]) -> int` | the label of the training image with the smallest squared_distance; ties: the earlier one |
| C3 (minimum path) | `accuracy(predictions: list[int], truth: list[int]) -> float` | the fraction of positions where they are equal, from 0.0 to 1.0; empty lists give 0.0 |
| C5 (not needed later) | `render_digit(image: list[int], width: int = 28, threshold: int = 128) -> str` | one line per row: '#' if pixel >= threshold, else '.'; lines joined with "\n", no newline at the end |

(minimum path) marks the tasks you need for Putting It All Together: C1 → C2 → C3. Each function has a stub cell further down, starting with `# ===== YOUR CODE HERE =====`: write your code there, run it, then run the cell below at any time to see where you are. Each line tells you the next step. (C4 is now in the OPTIONAL part.)

In [ ]:
# ▶️ JUST RUN · where am I?
check_all();   # the ; only stops Jupyter from printing the same result again as a dict

## Load the data

We'll work with 1,000 examples of handwritten digits and another 100 images to test our program.

Garabato's scanner turns every handwritten digit into a **28 × 28 grid of numbers**: 0 is paper, 255 is ink.
Python keeps each image as one list of 784 ints, and the labels (the digit that was written) in a second list,
in the same order. Curious? Open `data/mnist_sample_20.csv` in VS Code: each row is a label followed by 784
numbers.

In [ ]:
# ▶️ JUST RUN · load the 1,000 training images and the 100 test images (about 0.1 s)
# limit= reads only the first rows of each file: that's all this lab needs
train_images, train_labels = load_digits("data/mnist_train_10k.csv.gz", limit=1000)
test_images, test_labels = load_digits("data/mnist_test_2k.csv.gz", limit=100)
print(f"{len(train_images):,} training images and {len(test_images)} test images, "
      f"{len(train_images[0])} pixels each.")

### An image is a list

Row `r`, column `c` lives at position `r * 28 + c`: row 0 fills positions 0–27, row 1 fills 28–55, and so on.

In [ ]:
# ▶️ JUST RUN · look inside one image
image = train_images[0]
print("label:", train_labels[0], "· pixels:", len(image), "· pixel at row 14, column 14:", image[14 * 28 + 14])
print("row 14 (28 pixels):", image[14 * 28 : 15 * 28])
show_digit(image, label=train_labels[0])     # a text picture, then a one-line description

### Getting to Know Our Data (about 5 minutes)

**An image is just a collection of numbers, and we can display it in different ways.** The three cells below
show the **same image** (the first training image) three ways. The display functions are provided and
already implemented: you only run them, and change one number.

1. **The numbers themselves:** part of the 28 × 28 grid, each value from 0 (paper) to 255 (ink).
2. **Black and white:** each pixel becomes `#` (ink) if it is at least the threshold, `.` (paper) otherwise.
   (The same rule as `ink_count` in W and `render_digit` in C5.)
3. **Greyscale:** one character per shade, from `' '` (paper) through `.:-=+*#%` to `@` (darkest ink).

In [ ]:
# ▶️ JUST RUN · view 1: the numbers (rows 4 to 13, columns 8 to 23 of the 28 x 28 grid)
image = train_images[0]
print("This image is labelled", train_labels[0])
show_numbers(image, rows=range(4, 14), cols=range(8, 24))

In [ ]:
# ▶️ JUST RUN · view 2: black and white. Then change THRESHOLD to 50, then 200, and run this cell again each time
THRESHOLD = 128
show_binary(image, threshold=THRESHOLD)      # '#' if pixel >= THRESHOLD, else '.'

In [ ]:
# ▶️ JUST RUN · view 3: greyscale, one character per shade
show_greyscale(image)

**Your turn (2 minutes).** In view 2, set `THRESHOLD = 50`, run the cell, then try `200`.

- Which threshold keeps the faint grey edges of the strokes? Which one loses part of the digit?
- In view 1, row 7, column 13 holds `160`. Is it `#` or `.` with a threshold of 50? Of 128? Of 200?
- Which view tells you the most? Which one would you show to a person, and which one to a program?

<details>
<summary><strong>Answers</strong></summary>

A low threshold (50) turns faint grey pixels into ink, so the strokes look thicker; a high one (200) keeps
only the darkest ink, so thin parts of the 8 start to break up. The pixel at row 7, column 13 (160) is `#`
with 50 and with 128, and `.` with 200. View 1 holds all the information, view 3 shows it to a person, and
view 2 throws some of it away: the picture is the same list of numbers each time.

</details>

### W · Worked example: Count the ink · ▶️ JUST RUN

Garabato's scanner should flag blank parcels. Count how many pixels of an image are ink.

**You'll practise:** counting with a loop and an if (Theory S2)

**Signature**

```python
def ink_count(image: list[int], threshold: int = 128) -> int
```

**Toy example**

```python
ink_count([0, 128, 255, 127])    # 128 and 255 are ink -> 2
```

Javier writes this one live with you. The code is already in the next cell: read it line by line, run it, then run its check.

In [ ]:
# ▶️ JUST RUN · the worked example: already written, so read it line by line, then run it
def ink_count(image: list[int], threshold: int = 128) -> int:
    """Return how many pixels of `image` are ink: value >= threshold.

    >>> ink_count([0, 128, 255, 127])
    2
    """
    count = 0                      # the running count starts at zero
    for pixel in image:            # visit every pixel once
        if pixel >= threshold:     # ink? (a pixel equal to the threshold counts)
            count += 1             # one more ink pixel
    return count                   # after the loop, not inside it

In [ ]:
# ▶️ JUST RUN · check your work
check("W", ink_count);

**Try it (1 minute).** Change `>=` to `>` in the cell with `def ink_count`, run that cell, then run the
check again: you get a **NOT YET** that shows the failing test (its inputs, the expected count and your
result) and a likely cause. Change it back and run both cells again. (The `;` at the end of a check line
only stops Jupyter from printing `True` or `False` under the report.)

In [ ]:
# ▶️ JUST RUN
print("Ink pixels in the first training image:", ink_count(train_images[0]))

### Warm-up: predict the output

For while others finish their setup. Read each cell, **write down what you think it prints**, then run it.
The answers are under the third cell.

In [ ]:
# ▶️ JUST RUN · warm-up 1 (predict first, then run)
pixels = [0, 255, 128, 64]
print(pixels[1], pixels[-1], len(pixels))

In [ ]:
# ▶️ JUST RUN · warm-up 2 (predict first, then run)
total = 0
for x in [3, 1, 4]:
    total += x * x
print(total)

In [ ]:
# ▶️ JUST RUN · warm-up 3 (predict first, then run)
counts = {}
for digit in [7, 1, 7, 7]:
    counts[digit] = counts.get(digit, 0) + 1
print(counts)

<details>
<summary><strong>Answers</strong></summary>

1. `255 64 4`: position 1 is the second item, `-1` is the last one, and there are 4 items.
2. `26`: the running total is 9, then 10, then 26 (3 × 3 + 1 × 1 + 4 × 4).
3. `{7: 3, 1: 1}`: one key per digit, and its value counts how often it appeared.

</details>

## CORE: the tasks everyone does

**C1 → C2 → C3 → Putting It All Together → C5.** The first four are the **minimum path**: reaching Putting It All Together is a
complete lab. C5 completes the CORE.

For each task: read the card, write your code in the cell that starts with `# ===== YOUR CODE HERE =====`
(replace `pass`), run that cell, then run the check under it. The check shows the tests it ran: what each
one checks, its inputs, the expected result and your result.

### C1 · How different are two images? · ✏️ WRITE CODE

To read a digit, the machine compares it with digits it already knows. First it needs one number that says how different two images are.

**You'll practise:** a running total over two lists (Theory S2) · **Typical time:** 6–10 min

**Minimum path: you need this for Putting It All Together.**

**Signature**

```python
def squared_distance(a: list[int], b: list[int]) -> int
```

**Toy example**

```python
squared_distance([0, 0], [3, 4])        # (0 - 3) ** 2 + (0 - 4) ** 2 = 9 + 16 = 25
squared_distance([1, 2, 3], [1, 2, 3])  # identical images: 0
```

**Why square the differences?** We need one number that says how different two pixels are. Two candidates:

| Pixel in image a | Pixel in image b | Absolute difference | Squared difference |
|---|---|---|---|
| 100 | 110 | 10 | 100 |
| 100 | 150 | 50 | 2,500 |
| 100 | 200 | 100 | 10,000 |

Both grow when the pixels differ more, and neither cares which image is brighter. Squaring gives
proportionally more weight to big differences: a difference 10 times bigger counts 100 times more. Adding up
the squared differences gives the **Euclidean distance** (the straight-line distance from geometry) without its
final square root. We don't need the root: we only compare which training image is closer, and a square root
never changes which of two distances is smaller.

**Stuck for 7 minutes after Hint 3?** Run `squared_distance = rescue("C1")` in a new cell, carry on, and come back to C1 later.

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).
def squared_distance(a: list[int], b: list[int]) -> int:
    """Return the squared distance between two images of equal length: the sum of (a[i] - b[i]) ** 2.

    >>> squared_distance([0, 0], [3, 4])
    25
    """
    # TODO 1: start a running total at 0
    # TODO 2: for each position i, add (a[i] - b[i]) ** 2 to the total
    # TODO 3: after the loop, return the total
    pass

In [ ]:
# ▶️ JUST RUN · check your work
check("C1", squared_distance);

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

You need one number that grows as you walk through both images: the *running total* pattern (Theory S2, "Loops"). At each position, how different are the two pixels?

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. Start `total` at 0.
2. For each position `i` from 0 to `len(a) - 1` (that's `for i in range(len(a)):`), add `(a[i] - b[i]) ** 2` to `total`.
3. After the loop (one indent to the left), return `total`.

</details>

<details>
<summary><strong>Hint 3 · Skeleton</strong></summary>

    total = ___
    for i in range(len(a)):
        total += (___ - ___) ** 2
    return ___

</details>

<details>
<summary><strong>Hint 4 · Ask an AI tutor (not for the answer)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a tutor and do NOT write code for me. Here is my function and the check output: [paste]. Ask me one question at a time that helps me find the problem myself."*

</details>

> **New tool: zip (optional)**
>
> `for x, y in zip(a, b):` walks two lists side by side: first `x = a[0]` and `y = b[0]`, then `a[1]` and
> `b[1]`, and so on. It can replace `for i in range(len(a)):` with `a[i]` and `b[i]`. Both are correct:
> use the one you can read.

### C2 · The closest picture wins · ✏️ WRITE CODE

Nearest neighbour: compare the new digit with every training image and copy the label of the closest one.

**You'll practise:** the best-so-far loop over two parallel lists (Theory S2–S3) · **Typical time:** 12–20 min

**Minimum path: you need this for Putting It All Together.**

**Signature**

```python
def predict_nearest(image: list[int], train_images: list[list[int]], train_labels: list[int]) -> int
```

**Toy example**

```python
# A tiny world with 1-pixel images. The new image is [6]; four training images and their labels:
predict_nearest([6], [[9], [4], [20], [0]], [7, 3, 8, 5])
# distances: 9, 4, 196, 36 -> the closest is [4], at position 1, and its label is 3 -> returns 3
# If two training images are equally close, the earlier one wins.
```

**Stuck for 7 minutes after Hint 3?** Run `predict_nearest = rescue("C2")` in a new cell, carry on, and come back to C2 later.

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).
def predict_nearest(image: list[int], train_images: list[list[int]], train_labels: list[int]) -> int:
    """Return the label of the training image closest to `image` (ties: the earlier one).

    >>> predict_nearest([6], [[9], [4], [20], [0]], [7, 3, 8, 5])
    3
    """
    # TODO 1: best_distance starts at float("inf") (bigger than any real distance); best_index at 0
    # TODO 2: for each position i, compute squared_distance(image, train_images[i])
    # TODO 3:     if it is smaller than best_distance, remember the distance and i
    # TODO 4: after the loop, return the LABEL at best_index (not the position)
    pass

In [ ]:
# ▶️ JUST RUN · check your work
check("C2", predict_nearest);

**Your first prediction** (run it once C2 passes). The cell below takes one test image and shows what your two
functions do with it: the three training images with the smallest `squared_distance`, each with its label and
its distance, the true label of the test image (the ground truth), and the label your `predict_nearest`
returns: the label of the closest image. If C1 or C2 isn't ready yet, the cell tells you what to do.

In [ ]:
# ▶️ JUST RUN · a test image, its true label, and its 3 closest training images, by YOUR functions
test_index = 0                        # optional: try another test image, e.g. 13 or 33
show_top3(test_images[test_index], test_labels[test_index], train_images, train_labels,
          distance_fn=squared_distance, predict_fn=predict_nearest)

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

This is the *best-so-far* pattern (Theory S2): walk through every training image and remember the closest one so far. Start with a best distance that any real distance beats: `float("inf")`, infinity.

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. Set `best_distance = float("inf")` and `best_index = 0`.
2. For each position `i` in `range(len(train_images))`, compute `distance = squared_distance(image, train_images[i])`.
3. If `distance < best_distance`, remember both: `best_distance = distance` and `best_index = i`.
4. After the loop, return `train_labels[best_index]`: the label, not the position.

</details>

<details>
<summary><strong>Hint 3 · Skeleton</strong></summary>

    best_distance = float("inf")
    best_index = 0
    for i in range(len(train_images)):
        distance = squared_distance(___, ___)
        if distance ___ best_distance:
            best_distance = ___
            best_index = ___
    return train_labels[___]

</details>

<details>
<summary><strong>Hint 4 · Ask an AI tutor (not for the answer)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a tutor and do NOT write code for me. Here is my function and the check output: [paste]. Ask me one question at a time that helps me find the problem myself."*

</details>

### C3 · How often is it right? · ✏️ WRITE CODE

One right answer could be luck. Garabato wants a number: the fraction of test digits your machine reads correctly.

**You'll practise:** counting, dividing, and guarding the empty case (Theory S2) · **Typical time:** 5–8 min

**Minimum path: you need this for Putting It All Together.**

**Signature**

```python
def accuracy(predictions: list[int], truth: list[int]) -> float
```

**Toy example**

```python
accuracy([1, 2, 3, 4], [1, 2, 8, 4])  # 3 of 4 predictions match the ground truth -> 0.75
accuracy([], [])                      # nothing to count -> 0.0 (not an error)
```

**Stuck for 7 minutes after Hint 3?** Run `accuracy = rescue("C3")` in a new cell, carry on, and come back to C3 later.

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).
def accuracy(predictions: list[int], truth: list[int]) -> float:
    """Return the fraction of predictions that equal the truth (0.0 to 1.0); empty lists give 0.0.

    >>> accuracy([1, 2, 3, 4], [1, 2, 8, 4])
    0.75
    """
    # TODO 1: if the lists are empty, return 0.0
    # TODO 2: count the positions i where predictions[i] == truth[i]
    # TODO 3: return that count divided by the number of predictions
    pass

In [ ]:
# ▶️ JUST RUN · check your work
check("C3", accuracy);

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

Two patterns you know: *counting* (add 1 when a condition holds, Theory S2) and dividing. One trap: dividing by zero when the lists are empty.

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. If `len(truth) == 0`, return `0.0`.
2. Start `correct` at 0. For each `i` in `range(len(truth))`, add 1 when `predictions[i] == truth[i]`.
3. Return `correct / len(truth)`: a single `/` gives a float such as 0.75.

</details>

<details>
<summary><strong>Hint 3 · Skeleton</strong></summary>

    if len(truth) == ___:
        return ___
    correct = 0
    for i in range(len(truth)):
        if ___ == ___:
            correct += 1
    return ___ / ___

</details>

<details>
<summary><strong>Hint 4 · Ask an AI tutor (not for the answer)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a tutor and do NOT write code for me. Here is my function and the check output: [paste]. Ask me one question at a time that helps me find the problem myself."*

</details>

### Putting It All Together · ▶️ JUST RUN

This is where your three functions work together, on **100 handwritten digits your program has never seen**:

1. `squared_distance` (C1) **computes the differences** between two images;
2. `predict_nearest` (C2) uses it to **find the most similar** training image for each test image, and copies
   its label;
3. `accuracy` (C3) **counts the correct predictions**: how many of those labels match the ground truth.

`evaluate()` (provided) runs those three steps for the 100 test images and prints the result. How many will your
program read correctly? Run it and find out. It takes a few seconds: the first line tells you how long, then a
dot appears for every 10 digits. (Kernel busy for more than a minute? Press **Interrupt**, the square button.
After a **Restart**, run the setup cell, *Load the data* and your `def` cells again before this one: click this
cell, then **Execute Above Cells** in VS Code (the arrow next to the cell's run button), or **Runtime > Run
before** in Colab.)

In [ ]:
# ▶️ JUST RUN · Putting It All Together: 100 test images, each compared with 1,000 training images, with your three functions
test_run = evaluate(predict_nearest, n_train=1000, n_test=100, accuracy_fn=accuracy)

Now the mistakes: each misread test digit next to the training image it was closest to.
**Would you have read them correctly?**

In [ ]:
# ▶️ JUST RUN · each misread test digit next to the training image it was closest to
show_mistakes(predict_nearest, n=3)

**Reaching Putting It All Together is a complete lab.** Your program compared 100 test images with 1,000 training
images: 100 × 1,000 × 784 = 78,400,000 subtractions, in pure Python. Next: C5 completes the CORE; then
the AI pit stop and Draw your own digit, with the whole class; then OPTIONAL and ADVANCED tasks.

### C5 · Draw it in text · ✏️ WRITE CODE

Ops want to see what the machine saw, in a plain-text log. Turn an image into lines of `#` (ink) and `.` (paper).

**You'll practise:** nested loops, r * width + c, and building strings (Theory S2–S3) · **Typical time:** 10–15 min

**Not needed later:** nothing after this task depends on it, so you can come back to it after class.

**Signature**

```python
def render_digit(image: list[int], width: int = 28, threshold: int = 128) -> str
```

**Toy example**

```python
render_digit([0, 200, 255, 10], width=2)
# a 2 x 2 image: row 0 is [0, 200] -> ".#" and row 1 is [255, 10] -> "#."
# returns ".#\n#."   (print() shows it as two lines)
```

This is view 2 of *Getting to Know Our Data* (`show_binary`), written by you: one character per pixel, returned as a string instead of printed.

**Stuck for 7 minutes after Hint 3?** Run `render_digit = rescue("C5")` in a new cell, carry on, and come back to C5 later.

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).
def render_digit(image: list[int], width: int = 28, threshold: int = 128) -> str:
    """Return a text picture: one line per row, '#' if pixel >= threshold, else '.'.

    The lines are joined with newlines, with no newline after the last one:
    >>> print(render_digit([0, 200, 255, 10], width=2))
    .#
    #.
    """
    # TODO 1: for each row r (there are len(image) // width rows): build a string of '#' and '.'
    # TODO 2:     pixel (r, c) is image[r * width + c]; '#' if it is >= threshold
    # TODO 3: join the row strings with "\n" (no newline at the end) and return the result
    pass

In [ ]:
# ▶️ JUST RUN · check your work
check("C5", render_digit);

In [ ]:
# ▶️ JUST RUN · a real test digit, drawn by YOUR render_digit
show_rendered(render_digit, test_images[0], label=test_labels[0])

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

Two loops, one inside the other (Theory S2): the outer one walks the rows `r`, the inner one the columns `c`. Pixel (r, c) is `image[r * width + c]`. Build one string per row, then join the rows.

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. Start an empty list: `lines = []`.
2. For each row `r` in `range(len(image) // width)`: start `line = ""`; for each column `c` in `range(width)`, add `"#"` if `image[r * width + c] >= threshold`, else `"."`; then `lines.append(line)`.
3. Return `"\n".join(lines)`: a newline between the lines, none at the end.

</details>

<details>
<summary><strong>Hint 3 · Skeleton</strong></summary>

    lines = []
    for r in range(len(image) // width):
        line = ""
        for c in range(___):
            if image[___ * width + ___] >= threshold:
                line += ___
            else:
                line += ___
        lines.append(___)
    return "\n".join(___)

</details>

<details>
<summary><strong>Hint 4 · Ask an AI tutor (not for the answer)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a tutor and do NOT write code for me. Here is my function and the check output: [paste]. Ask me one question at a time that helps me find the problem myself."*

</details>

**The mistakes of Putting It All Together, drawn by your `render_digit`** (run Putting It All Together first).

In [ ]:
# ▶️ JUST RUN · the same mistakes as above, drawn by YOUR render_digit
show_mistakes(predict_nearest, n=2, renderer=render_digit)

**CORE complete.** Well done. Next, with the whole class: the AI pit stop and drawing your own digit. Then
any OPTIONAL task, or ADVANCED if you move faster.

## After the CORE: two activities with the whole class

Javier runs these two with everyone, so you can join them wherever you are in the CORE.

### AI pit stop: read a traceback · ▶️ JUST RUN (about 5 minutes)

When Python stops with an error, it prints a **traceback**. Read it **from the bottom up**: the last line
names the error, the lines just above it show where it happened, and the lines higher up show which code
called that code.

The cell below runs a small function with a bug. It catches the error so that *Run All* keeps going, and
prints the traceback as Python would.

1. **Read it bottom-up.** What is the error? In which function, on which line? Which value of `i` was too big?
2. **Ask an AI to explain it.** Paste the traceback with: *"Explain this Python traceback to a beginner.
   Don't fix the code: tell me what the error means and which line to look at."* (The cell shows your
   user folder as `~`, because a folder name is often a person's name. Still, read what you paste
   first: never paste personal data into an AI tool.)
3. **Verify by changing exactly one thing** in the cell, then run it again. Did the error go away for the
   reason the AI gave?

In [ ]:
# ▶️ JUST RUN first (it prints a traceback on purpose). Step 3: change exactly one thing, then run it again.
import sys
import traceback
from labkit.text import hide_user_folder

def brightest_pixel(image: list[int]) -> int:
    """Return the largest pixel value of an image."""
    best = image[0]
    for i in range(len(image) + 1):
        if image[i] > best:
            best = image[i]
    return best

def brightness_report(images: list[list[int]]) -> None:
    """Print the brightest pixel of each image."""
    for n in range(len(images)):
        print(f"image {n}: brightest pixel {brightest_pixel(images[n])}")

try:
    brightness_report(test_images[:3])
except Exception:
    # print the traceback as Python would (your user folder shows as ~), then carry on
    print(hide_user_folder(traceback.format_exc()), file=sys.stderr)

<details>
<summary><strong>What we found (open after you've tried)</strong></summary>

The last line says `IndexError: list index out of range`. The line above it is
`if image[i] > best:` inside `brightest_pixel`, which was called by `brightness_report`.
`range(len(image) + 1)` goes one step too far: an image has positions 0 to 783, so `image[784]` doesn't
exist. The one change: `range(len(image))`. A good AI explanation points at the same line; checking it
by changing one thing is how you know it was right.

</details>

### Draw your own digit (whole class, about 8 minutes)

Does your machine read **your** handwriting? Whichever route you take, your drawing goes through the same
pipeline:

```text
your drawing        ->  numbers               ->  a 28 x 28 image          ->  a prediction
'#' and '.'             255 for ink, 0 paper      784 numbers, centred         predict_nearest(...)
(or the mouse)          (as in view 1 above)      like an MNIST digit          says: 7
```

Remember *Getting to Know Our Data*? An image is just a list of numbers. Your drawing becomes one too, in
exactly the format our program needs: 784 numbers from 0 to 255.

**What `sketch_to_image()` does with a keyboard drawing** (provided, already implemented):

1. **Each character becomes a number:** `#` becomes 255 (ink) and `.` becomes 0 (paper).
2. **It expands the drawing:** your sketch is 14 × 14 characters, but MNIST images are 28 × 28 pixels. So
   each character becomes a block of 2 × 2 pixels: 14 × 2 = 28. That is what "expanding" means.
3. **It centres the digit** in the 28 × 28 frame, because every MNIST digit is centred.
4. **It softens the edges:** pixels next to the ink get a grey value, like a pen stroke on paper.

If C2 isn't passing yet, run `predict_nearest = rescue("C2")` in a new cell first.

#### Route 1 · the keyboard (always works)

Edit the sketch in the next cell: `#` is ink, `.` is paper, at most 14 rows of 14 characters. Then run it.
It shows your digit as numbers (like view 1), as a picture, and what your machine says.

In [ ]:
# ✏️ WRITE CODE · edit the sketch (# = ink, . = paper; at most 14 rows of 14), then run this cell
sketch = """
..............
..............
...########...
.........##...
.........##...
........##....
........##....
.......##.....
.......##.....
......##......
......##......
..............
..............
..............
"""
image = sketch_to_image(sketch)          # characters -> numbers -> 28 x 28 (expanded x 2, centred, softened)
show_numbers(image)                      # part of your drawing as numbers: 255 ink, 0 paper, grey edges
show_digit(image)
guess = try_predict(predict_nearest, image, train_images, train_labels)   # if C1/C2 aren't ready: what to do
if guess is not None:
    print("Your machine says:", guess)

#### Route 2 · the canvas: draw with the mouse

The kit includes a small drawing page, `draw_digit.html`, in the `tools` folder of the kit folder (the folder
with this notebook and `labkit`). It works offline, in your web browser.

1. **Find the file.** Open your file explorer (Windows: File Explorer; macOS: Finder), go to the kit folder,
   then into its `tools` folder. (In VS Code you can also right-click `tools/draw_digit.html` in the Explorer
   side bar and choose **Reveal in File Explorer** or **Reveal in Finder**.)
2. **Open it in your browser:** double-click `draw_digit.html`. If it opens in another program, right-click it
   and choose **Open with** > your browser (Edge, Chrome, Firefox or Safari).
3. **Draw one digit** in the box with the mouse (a finger or a pen work too). Make it big: fill most of the box.
4. **Click "Copy 784 numbers".** If the page says it couldn't copy, it selects the numbers for you: press
   Ctrl+C (Cmd+C on a Mac).
5. **Come back to this notebook**, click between the two lines of triple quotes `"""` in the next cell, and
   paste (Ctrl+V, or Cmd+V on a Mac).
6. **Run the cell.** Nothing pasted yet? It uses an example drawing that comes with the kit.

The canvas gives you the raw drawing. `prepare_drawing()` (provided) then does what the MNIST authors did to
every digit: it crops the drawing to the ink, scales it so the longer side is 20 pixels, and centres it.

In [ ]:
# ✏️ WRITE CODE · paste the 784 numbers from tools/draw_digit.html between the two sets of triple quotes, then run
drawing = """
"""
from pathlib import Path
from labkit import parse_drawing, side_by_side

if not drawing.strip():          # nothing pasted yet: use the example drawing that comes with the kit
    drawing = Path("tools/drawing_example.txt").read_text(encoding="utf-8")
    print("Nothing pasted yet, so this cell uses tools/drawing_example.txt (a 7, drawn small and off-centre).")
raw = parse_drawing(drawing)                # the 784 numbers, exactly as drawn
image = prepare_drawing(drawing)            # crop to the ink, scale to 20 x 20, centre it, like MNIST
side_by_side([raw, image], ["as drawn", "after prepare_drawing"])
as_drawn = try_predict(predict_nearest, raw, train_images, train_labels)
prepared = try_predict(predict_nearest, image, train_images, train_labels) if as_drawn is not None else None
if prepared is not None:
    print(f"Your machine says: {as_drawn} for the drawing as drawn, {prepared} after prepare_drawing")

**Compare the two answers of the canvas route.** With the example drawing they differ: 5 as drawn, 7 after
`prepare_drawing`. Nearest neighbour compares pixel by pixel, so a digit that is too small or in the wrong
place overlaps the wrong training images. The training digits were scanned, scaled and centred; a raw drawing
isn't. **Training data that looks different from real input** is one of the most common reasons a model
fails in the real world. Then try to fool it anyway: a 4 with a closed top, a 7 with a bar through the
middle, a very thick or very thin stroke.

## OPTIONAL: extra practice

Extra practice for when your CORE is done. Pick any task, in any order: each one is independent, has a `check()` and hints, and nothing later depends on it.

| Task | What you'll learn | Difficulty | Typical time |
|---|---|---|---|
| **C4** Is our test fair? | counting with a dict, and what a balanced dataset is | gentle | 5–8 min |
| **ENG1** Check Your Inputs: Are Both Images the Same Size? | data validation: detect wrong inputs and raise an error that explains them | gentle | 10–15 min |
| **SCI1** The average digit | one average image per digit: 10 comparisons instead of 1,000, and what that costs | moderate | 12–15 min |
| **BLD1** Where it fails | which digits your program confuses most, from your own results | moderate | 12–15 min |

In [ ]:
# ▶️ JUST RUN · tools for the OPTIONAL and ADVANCED tasks (run this once)
import time
from labkit import downsample, league_run, parse_drawing, side_by_side, stopwatch
from labkit.lab1_tools import ready, split_train

### OPTIONAL · C4 · Is our test fair? · ✏️ WRITE CODE

If the 100 test digits were mostly 1s, a high score would mean little. Count how many test images there are of each label.

**What you'll learn:** counting with a dict, and checking whether a dataset is balanced (Theory S2–S3) · **Difficulty:** gentle · **Typical time:** 5–8 min

**Signature**

```python
def count_labels(labels: list[int]) -> dict[int, int]
```

**Toy example**

```python
count_labels([3, 1, 3])    # {3: 2, 1: 1}
count_labels([])           # {}
```

**What is a balanced dataset?** A dataset is *balanced* when every class (here: every digit from 0 to 9)
appears about equally often. Why it matters: imagine that 80% of the test images were 1s. A program that always
answers 1, without even looking at the image, would score 80% while recognising nothing at all. So before
trusting an accuracy, look at the class distribution: how many images of each digit are there? Balance is one
way to check the quality of an evaluation, not a guarantee: a balanced test can still be too small, or too
different from the digits the program will meet in real use.

**Stuck for 7 minutes after Hint 3?** Run `count_labels = rescue("C4")` in a new cell, carry on, and come back to C4 later.

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).
def count_labels(labels: list[int]) -> dict[int, int]:
    """Return how many times each label appears, as {label: count}.

    >>> count_labels([3, 1, 3])
    {3: 2, 1: 1}
    """
    # TODO 1: start with an empty dict
    # TODO 2: for each label: add 1 to its count (a label you haven't seen yet starts at 0)
    # TODO 3: return the dict
    pass

In [ ]:
# ▶️ JUST RUN · check your work
check("C4", count_labels);

In [ ]:
# ▶️ JUST RUN · how many test images of each digit? Is our test fair? (needs C4)
if ready(count_labels):
    counts = count_labels(test_labels)
    for digit in sorted(counts):
        print(f"digit {digit}: {counts[digit]:3d} images  {'#' * counts[digit]}")
    biggest = max(counts.values()) / len(test_labels)
    print(f"The most common digit is {biggest:.0%} of the test set: always answering it would score "
          f"{biggest:.0%}.")

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

Counting with a dict (Theory S3): one key per label, and its value is how many you've seen so far. The first time you see a label, it isn't a key yet.

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. Start with an empty dict: `counts = {}`.
2. For each label (`for label in labels:`, or `for i in range(len(labels)):` with `label = labels[i]`): if `label` is already a key, add 1 to its value; otherwise set it to 1. (`counts[label] = counts.get(label, 0) + 1` does both in one line.)
3. Return `counts`.

</details>

<details>
<summary><strong>Hint 3 · Skeleton</strong></summary>

    counts = {}
    for label in labels:
        if label in counts:
            counts[label] = ___
        else:
            counts[label] = ___
    return ___

</details>

<details>
<summary><strong>Hint 4 · Ask an AI tutor (not for the answer)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a tutor and do NOT write code for me. Here is my function and the check output: [paste]. Ask me one question at a time that helps me find the problem myself."*

</details>

### OPTIONAL · ENG1 · Check Your Inputs: Are Both Images the Same Size? · Engineer track · ✏️ WRITE CODE

squared_distance trusts its inputs: give it two images of different sizes and it fails deep inside the loop, or quietly gives a wrong number. Make a version that checks first.

**What you'll learn:** data validation and error handling: check the inputs before any work, and raise an error that explains the problem · **Difficulty:** gentle · **Typical time:** 10–15 min

**Signature**

```python
def safe_distance(a: list[int], b: list[int]) -> int
```

**Toy example**

```python
safe_distance([0, 0], [3, 4])    # 25, exactly like squared_distance
safe_distance([1, 2], [1])       # ValueError: images have different lengths: 2 vs 1
```

**Stuck for 7 minutes after Hint 3?** Run `safe_distance = rescue("ENG1")` in a new cell, carry on, and come back to ENG1 later.

> **New tool: raise**
>
> `raise ValueError("message")` stops the function at once and reports the problem, with your message, to
> whoever called it. `ValueError` means "an argument has the right type but a wrong value". For example:
>
> ```python
> if threshold < 0:
>     raise ValueError(f"threshold must be 0 or more, not {threshold}")
> ```

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).
def safe_distance(a: list[int], b: list[int]) -> int:
    """Like squared_distance, but raise ValueError("images have different lengths: 784 vs 783")
    when the two images have different lengths.

    >>> safe_distance([0, 0], [3, 4])
    25
    """
    # TODO 1: if the lengths differ, raise ValueError with exactly the message from the card
    # TODO 2: otherwise return the squared distance (you may call squared_distance)
    pass

In [ ]:
# ▶️ JUST RUN · check your work
check("ENG1", safe_distance);

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

Check the inputs first, before any work: compare `len(a)` and `len(b)`. If they differ, `raise` a ValueError (see the New tool box).

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. `if len(a) != len(b):` raise a ValueError whose message is an f-string built from `len(a)` and `len(b)`; it must read exactly like the card: `images have different lengths: 2 vs 1`.
2. Otherwise return the squared distance: call `squared_distance(a, b)` (or copy its loop).

</details>

<details>
<summary><strong>Hint 3 · Ask an AI reviewer (keep only what you can explain)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a code reviewer and do NOT rewrite my code. Here is my function and what the check or my test printed: [paste]. Point out one problem at a time and ask me how I would fix it."*

</details>

In [ ]:
# ▶️ JUST RUN · see your error in action: a real image against one with its last pixel missing
if ready(safe_distance):
    try:
        result = safe_distance(test_images[0], test_images[1][:783])
        print("No error raised: safe_distance returned", result)
    except Exception as error:
        print(f"{type(error).__name__}: {error}")

### OPTIONAL · SCI1 · The average digit · Scientist track · ✏️ WRITE CODE

Nearest neighbour compares each digit with 1,000 images. What if each digit had one 'average image', and we compared with just 10?

**What you'll learn:** averages with nested loops and a dict of lists, and the trade-off between simplicity, speed and accuracy (Theory S2–S3) · **Difficulty:** moderate · **Typical time:** 12–15 min

**Signature**

```python
def fit_centroids(images: list[list[int]], labels: list[int]) -> dict[int, list[float]]
def predict_centroid(image: list[int], centroids: dict[int, list[float]]) -> int
```

**Toy example**

```python
fit_centroids([[0, 2], [4, 7], [10, 10]], [1, 1, 7])     # {1: [2.0, 4.5], 7: [10.0, 10.0]}
predict_centroid([1, 1], {0: [0.0, 0.0], 5: [3.0, 3.0], 9: [1.0, 1.5]})    # 9, the closest average
```

**The trade-off.** Nearest neighbour compares every new digit with **1,000 reference images**. The
average-digit reader first builds **one average image per digit** (once), then compares every new digit with
only **10 averages**: 100 times fewer comparisons. That is simpler and much faster, but an average blurs the
different ways people write the same digit, so expect to lose some accuracy. The last cell of this task
measures all three: simplicity (10 images to keep instead of 1,000), speed and accuracy.

Two functions, one check: `check("SCI1", fit_centroids)` checks `fit_centroids` first, then
`predict_centroid`.

**Stuck for 7 minutes after Hint 2?** Run `fit_centroids = rescue("SCI1")` and `from labkit.reference.lab1 import predict_centroid` in a new cell, carry on, and come back to SCI1 later.

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).
def fit_centroids(images: list[list[int]], labels: list[int]) -> dict[int, list[float]]:
    """Return the average image of each label: {label: list of floats}.

    >>> fit_centroids([[0, 2], [4, 7], [10, 10]], [1, 1, 7])
    {1: [2.0, 4.5], 7: [10.0, 10.0]}
    """
    # TODO 1: for each label, a running total per pixel (a NEW list of zeros) and a count
    # TODO 2: add every image into the totals of its label; count it
    # TODO 3: divide every total by its label's count (with /) and return {label: averages}
    pass

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).
def predict_centroid(image: list[int], centroids: dict[int, list[float]]) -> int:
    """Return the label whose average image is closest to `image` (the smallest squared distance).

    >>> predict_centroid([1, 1], {0: [0.0, 0.0], 5: [3.0, 3.0], 9: [1.0, 1.5]})
    9
    """
    # TODO: the best-so-far loop of C2, over `for label in centroids:` instead of the training images
    pass

In [ ]:
# ▶️ JUST RUN · check your work
check("SCI1", fit_centroids);   # checks fit_centroids, then predict_centroid

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

fit: for each label keep a running total *per pixel* (a list of zeros as long as an image) and a count; at the end divide every total by the count. predict: C2's best-so-far loop again, over 10 averages instead of 1,000 images.

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. `fit_centroids`: `totals = {}` and `counts = {}`. For each image and its label: if the label is new, `totals[label] = [0] * len(image)` and `counts[label] = 0`; add each pixel into `totals[label]`; add 1 to `counts[label]`. Then, for each label, divide every total by `counts[label]` with `/`.
2. `predict_centroid`: `for label in centroids:` compute `squared_distance(image, centroids[label])` and keep the label with the smallest distance.

</details>

<details>
<summary><strong>Hint 3 · Ask an AI reviewer (keep only what you can explain)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a code reviewer and do NOT rewrite my code. Here is my function and what the check or my test printed: [paste]. Point out one problem at a time and ask me how I would fix it."*

</details>

In [ ]:
# ▶️ JUST RUN · the 10 "ghost digits": one average per digit, learnt from our 1,000 training images
# (shrunk to 14 x 14 so that three fit side by side)
if ready(fit_centroids):
    centroids = fit_centroids(train_images, train_labels)
    for first in (0, 3, 6, 9):
        digits = list(range(first, min(first + 3, 10)))
        side_by_side([downsample([round(v) for v in centroids[d]]) for d in digits],
                     [f"average {d}" for d in digits])

In [ ]:
# ▶️ JUST RUN · simplicity, speed and accuracy, on the same 100 test images: 1,000 comparisons per digit vs 10
# (about 1 s; about 6 s more if Putting It All Together hasn't run yet)
if ready(fit_centroids, predict_centroid):
    rows = []
    nn = globals().get("test_run")
    if not isinstance(nn, dict) or nn.get("status") != "done":
        nn = evaluate(predict_nearest, n_train=1000, n_test=100)
    if nn.get("status") == "done":
        rows.append(("Nearest neighbour", "1,000 images", nn["correct"] / nn["total"],
                     nn["seconds"] / nn["total"]))
    start = time.perf_counter()
    centroids = fit_centroids(train_images, train_labels)
    fit_seconds = time.perf_counter() - start
    start = time.perf_counter()
    guesses = [predict_centroid(image, centroids) for image in test_images]
    seconds = time.perf_counter() - start
    right = sum(1 for guess, truth in zip(guesses, test_labels) if guess == truth)
    rows.append(("Average digits", f"{len(centroids)} averages", right / len(test_labels),
                 seconds / len(test_labels)))
    print(f"{'Reader':<20}{'Compares with':<16}{'Accuracy':>9}{'Time per digit':>16}")
    for name, compares, acc, per_digit in rows:
        print(f"{name:<20}{compares:<16}{acc:>9.1%}{1000 * per_digit:>13.2f} ms")
    print(f"(Building the averages took {fit_seconds:.2f} s, once. They keep {len(centroids)} images "
          f"instead of {len(train_images):,}.)")

**Think about it.** How much faster is the average-digit reader, and how much accuracy does it lose? Which
would you choose for Garabato's scanner, which reads thousands of parcels an hour? Which digits do you
think the averages blur the most? (Look at the ghost digits.)

### OPTIONAL · BLD1 · Where it fails · Builder track · ✏️ WRITE CODE

Which digits does your machine confuse, and with what? Count the mistakes by (truth, prediction) pair, then list the most common ones.

**What you'll learn:** which digits your program confuses most, using your own results: tuples as dict keys and sorting with a key (Theory S3) · **Difficulty:** moderate · **Typical time:** 12–15 min

**Signature**

```python
def confusion_pairs(predictions: list[int], truth: list[int]) -> dict[tuple[int, int], int]
def top_confusions(pairs: dict[tuple[int, int], int], n: int = 3) -> list[tuple[tuple[int, int], int]]
```

**Toy example**

```python
confusion_pairs([7, 7, 1, 3, 7], [1, 1, 1, 8, 7])    # {(1, 7): 2, (8, 3): 1}: a 1 read as 7 twice, an 8 read as 3 once
top_confusions({(2, 7): 2, (8, 3): 1, (4, 9): 5, (1, 7): 2})
# [((4, 9), 5), ((1, 7), 2), ((2, 7), 2)]: the biggest count first; equal counts: the smaller pair first
```

Two functions, one check: `check("BLD1", confusion_pairs)` checks `confusion_pairs` first, then `top_confusions`. Sorting with a key is a new tool: see the box below.

**Stuck for 7 minutes after Hint 2?** Run `confusion_pairs = rescue("BLD1")` and `from labkit.reference.lab1 import top_confusions` in a new cell, carry on, and come back to BLD1 later.

> **New tool: sorted with a key**
>
> `sorted(items, key=...)` sorts by whatever the key function returns for each item. A key can return a
> tuple: it compares the first parts, then the second parts for ties. A minus sign flips numbers around:
>
> ```python
> words = ["fig", "apple", "kiwi", "plum"]
> sorted(words, key=lambda w: (len(w), w))      # ['fig', 'kiwi', 'plum', 'apple']: shortest first, then a-z
> sorted(words, key=lambda w: (-len(w), w))     # ['apple', 'kiwi', 'plum', 'fig']: longest first, then a-z
> ```
>
> `lambda w: (len(w), w)` is a one-line function with no name: it takes `w` and returns `(len(w), w)`. A
> normal `def by_length(w): return (len(w), w)`, then `key=by_length`, does the same.

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).
def confusion_pairs(predictions: list[int], truth: list[int]) -> dict[tuple[int, int], int]:
    """Count the mistakes as {(true label, predicted label): how many times}; right answers are skipped.

    >>> confusion_pairs([7, 7, 1, 3, 7], [1, 1, 1, 8, 7])
    {(1, 7): 2, (8, 3): 1}
    """
    # TODO: C4's counting pattern, with the key (truth[i], predictions[i]), only where they differ
    pass

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).
def top_confusions(pairs: dict[tuple[int, int], int], n: int = 3) -> list[tuple[tuple[int, int], int]]:
    """Return the n most frequent mistakes as [((true, predicted), count), ...].

    The biggest count first; for equal counts, the smaller pair first.
    """
    # TODO: list(pairs.items()) gives [(pair, count), ...]: sort it (see the New tool box), keep n
    pass

In [ ]:
# ▶️ JUST RUN · check your work
check("BLD1", confusion_pairs);   # checks confusion_pairs, then top_confusions

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

confusion_pairs is C4 again, with a tuple `(truth, prediction)` as the key, and only for wrong answers. top_confusions: make a list with `list(pairs.items())`, sort it, keep the first n.

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. `confusion_pairs`: for each `i`, if `predictions[i] != truth[i]`, build `pair = (truth[i], predictions[i])` and count it like C4.
2. `top_confusions`: `items = list(pairs.items())`; sort with a key that puts the biggest count first and, for equal counts, the smaller pair first (the New tool box shows how a tuple key and a minus sign do that); return `items[:n]`.

</details>

<details>
<summary><strong>Hint 3 · Ask an AI reviewer (keep only what you can explain)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a code reviewer and do NOT rewrite my code. Here is my function and what the check or my test printed: [paste]. Point out one problem at a time and ask me how I would fix it."*

</details>

In [ ]:
# ▶️ JUST RUN · part 1: your machine's mistakes in Putting It All Together (100 test images)
if ready(confusion_pairs, top_confusions):
    if "test_run" not in globals() or not test_run["predictions"]:
        print("This uses the predictions of Putting It All Together: run that cell first.")
    else:
        pairs = confusion_pairs(test_run["predictions"], test_run["labels"])
        print(f"Putting It All Together: {sum(pairs.values())} mistakes, in {len(pairs)} different (truth, read as) pairs.")
        print("Top 3:", top_confusions(pairs, n=3))

With only 100 test images, most pairs happen once or twice: too few to see a pattern. Part 2 uses a
bigger sample: the fast average-digit reader (the reference version from SCI1) on every test image in the
kit (2,000).

In [ ]:
# ▶️ JUST RUN · part 2: the average-digit reader on all 2,000 test images (about 3 s), one example per top pair
from labkit.reference import lab1 as reference
if ready(confusion_pairs, top_confusions):
    all_train_images, all_train_labels = load_digits("data/mnist_train_10k.csv.gz")
    all_test_images, all_test_labels = load_digits("data/mnist_test_2k.csv.gz")
    averages = reference.fit_centroids(all_train_images, all_train_labels)
    said = [reference.predict_centroid(image, averages) for image in all_test_images]
    pairs = confusion_pairs(said, all_test_labels)
    print(f"Average-digit reader: {sum(pairs.values())} mistakes in 2,000 test images, {len(pairs)} different pairs.")
    for (truth, read_as), count in top_confusions(pairs, n=3):
        first = next(i for i in range(len(said)) if all_test_labels[i] == truth and said[i] == read_as)
        print(f"\nA {truth} read as {read_as}: {count} times. One of them, test image #{first}:")
        show_digit(all_test_images[first])

## ADVANCED: for students who move faster

For students who move faster. These tasks are more open: a sanity check plus a cell where you explain your result. Nothing here is needed for the next lab.

| Task | What you'll learn | Difficulty | Typical time |
|---|---|---|---|
| **ENG2** Make it faster | early exit with break, and measuring speed honestly | demanding | 15–25 min |
| **SCI2** Let the Nearest Images Vote | k nearest neighbours and a majority vote; choosing k honestly on validation data | demanding | 20–25 min |
| **BLD2** Robust drawings | crop, scale and centre a drawing with 2-D indexing (considerably more work than BLD1) | demanding | 20–25 min |
| **Challenge** the Pure-Python League | build the best digit reader you can, within a time budget; or a PNG encoder, or the numpy division | open-ended | open-ended |

In [ ]:
# ▶️ JUST RUN · tools for the ADVANCED tasks (the same as at the start of OPTIONAL: run this once)
import time
from labkit import downsample, league_run, parse_drawing, side_by_side, stopwatch
from labkit.lab1_tools import ready, split_train

### ADVANCED · ENG2 · Make it faster · Engineer track · ✏️ WRITE CODE

Putting It All Together did 78 million subtractions, and most were wasted: once a running total reaches the best distance so far, that training image can't win any more.

**What you'll learn:** early exit with break, and measuring speed honestly · **Difficulty:** demanding · **Typical time:** 15–25 min

**Signature**

```python
def predict_nearest_fast(image: list[int], train_images: list[list[int]], train_labels: list[int], stats: dict | None = None) -> int
```

**Toy example**

```python
# The same answers as predict_nearest, sooner. With a dict, it also counts what it skipped:
stats = {}
predict_nearest_fast(test_images[0], train_images, train_labels, stats)
stats["skipped"]     # how many pixel comparisons were never needed
```

Keep the 4th parameter `stats=None`: the experiment cell below passes a dict, and your function adds the pixels it skipped to `stats["skipped"]`. The check is a sanity check: same answers as `predict_nearest` (ties too), and some pixels skipped.

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).

def predict_nearest_fast(image: list[int], train_images: list[list[int]], train_labels: list[int],
                         stats: dict | None = None) -> int:
    """Like predict_nearest, but stop adding up a distance as soon as it can't win.

    If `stats` is a dict, add the number of pixel comparisons you skipped to stats["skipped"].
    """
    # TODO: copy your predict_nearest, but write the distance loop here instead of calling squared_distance.
    #       After each step of the running total: if it is >= best_distance, this image can't win: break.
    #       Update the best only when the loop went through every pixel.
    pass

In [ ]:
# ▶️ JUST RUN · check your work
check("ENG2", predict_nearest_fast);

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

Inside the pixel loop, right after `total += ...`, compare the running total with `best_distance`. If it is already `>=`, `break`: stop this image and go to the next one.

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. Start like predict_nearest: `best_distance = float("inf")`, `best_index = 0`, a loop over the training images.
2. For each one, add up the squared differences pixel by pixel in an inner loop. After each step: if `total >= best_distance`, add `len(image) - p - 1` to `stats["skipped"]` (only when `stats` is not None) and `break`.
3. Update `best_distance` and `best_index` only when the inner loop finished without `break` (a flag such as `finished = True`, set to False before `break`, tells you).

</details>

<details>
<summary><strong>Hint 3 · Ask an AI reviewer (keep only what you can explain)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a code reviewer and do NOT rewrite my code. Here is my function and what the check or my test printed: [paste]. Point out one problem at a time and ask me how I would fix it."*

</details>

In [ ]:
# ▶️ JUST RUN · the experiment: plain vs fast on the 100 test images of Putting It All Together (about 10 s)
takes_stats = predict_nearest_fast.__code__.co_argcount >= 4      # the card's 4th parameter, stats=None
if not takes_stats:
    print("predict_nearest_fast needs its 4th parameter, stats=None: this cell passes it a dict (see the card).")
elif ready(predict_nearest_fast, predict_nearest):
    stopwatch.clear()
    stats = {"skipped": 0}
    with stopwatch("predict_nearest"):
        plain = [predict_nearest(image, train_images, train_labels) for image in test_images]
    with stopwatch("predict_nearest_fast"):
        fast = [predict_nearest_fast(image, train_images, train_labels, stats) for image in test_images]
    stopwatch.table()
    same = sum(1 for p, q in zip(plain, fast) if p == q)
    work = len(test_images) * len(train_images) * 784
    print(f"Identical predictions: {same} of {len(plain)}")
    print(f"Pixel comparisons skipped: {stats['skipped']:,} of {work:,} ({stats['skipped'] / work:.0%})")

In [ ]:
# ▶️ JUST RUN (optional) · a speed ladder: one distance between two real images, written four ways (under 1 s).
# The four functions are provided and already implemented: don't modify them. Which is fastest, and why?
import math
import time

def loop_power(a, b):                 # the plan of Hint 2
    total = 0
    for i in range(len(a)):
        total += (a[i] - b[i]) ** 2
    return total

def loop_multiply(a, b):              # the same sum, multiplying instead of ** 2
    total = 0
    for i in range(len(a)):
        difference = a[i] - b[i]
        total += difference * difference
    return total

def loop_with_check(a, b, best=float("inf")):    # plus the test the fast version makes at every step
    total = 0
    for i in range(len(a)):
        difference = a[i] - b[i]
        total += difference * difference
        if total >= best:
            break
    return total

def built_in_dist(a, b):              # math.dist runs in C; squared and rounded back to an int
    return round(math.dist(a, b) ** 2)

a, b = test_images[0], train_images[0]
for distance_fn in (loop_power, loop_multiply, loop_with_check, built_in_dist):
    assert distance_fn(a, b) == loop_power(a, b)       # four ways, one answer
    start = time.perf_counter()
    for _ in range(200):
        distance_fn(a, b)
    print(f"{distance_fn.__name__:<16}{(time.perf_counter() - start) / 200 * 1e6:8.1f} µs per distance")

### Explain your result (ENG2)

Edit this cell. A few sentences are enough.

- How much faster was it, and were the predictions identical?
- It skipped a big share of the comparisons, so why isn't it that much faster? The speed ladder above
  has data: what does the extra `if` at every step cost?
- Which way of writing one distance is fastest, and why? (Python runs a loop one small step at a time;
  `math.dist` runs the whole loop in C.)
- Which training images would you compare first to skip even more? (Think about the label you expect.)

### ADVANCED · SCI2 · Let the Nearest Images Vote · Scientist track · ✏️ WRITE CODE

Why trust only the single closest image? Ask several similar images and let them vote.

**What you'll learn:** k nearest neighbours: sorting (distance, position) pairs, a majority vote with a dict, and choosing k honestly on validation data · **Difficulty:** demanding · **Typical time:** 20–25 min

#### The idea, without code

Nearest neighbour trusts **one** image: the closest. If that image happens to be an unusual 7 that looks
like a 1, the answer is 1, even when the next few closest images are all 7s.

Instead, **ask the k closest images and let them vote**: with k = 5, take the 5 most similar training
images; each one votes for its own label, and the label with the most votes wins. One odd neighbour gets
outvoted. If the vote is a tie (for example 2 votes for 7 and 2 for 1), trust the closest of the tied
neighbours.

Which k is best? A small k listens to very few images (one odd neighbour can decide); a big k also lets
images that are far away vote. There is no rule: it is an **experiment**. We try several values of k on
*validation* images (taken from the training data) and look at the test images only once, at the end, so
the test score stays honest.

#### The implementation

`predict_knn` does the vote. The check is a sanity check (the toy cases, and k = 1 equals nearest
neighbour). The real work is the experiment: `split_train` (provided) keeps 200 *training* images apart for
validation, you pick k there, and only then score the test set, once.

```python
def predict_knn(image: list[int], train_images: list[list[int]], train_labels: list[int], k: int = 3) -> int

predict_knn([5], [[0], [4], [6], [7], [30]], [2, 1, 9, 9, 1], k=3)   # the 3 nearest say 1, 9, 9 -> 9
predict_knn([5], [[4], [7], [0]], [2, 6, 3], k=2)                    # a 1-1 vote: the closer one, 2, wins
```

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).

def predict_knn(image: list[int], train_images: list[list[int]], train_labels: list[int], k: int = 3) -> int:
    """Return the majority label of the k nearest training images (a tied vote: the closest tied neighbour)."""
    # TODO 1: a list of (distance, position) pairs for every training image; sort it (nearest first)
    # TODO 2: count the labels of the first k pairs with a dict
    # TODO 3: walk the first k pairs again from the nearest; return the first label with the top count
    pass

In [ ]:
# ▶️ JUST RUN · check your work
check("SCI2", predict_knn);

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

Make a list of `(distance, position)` pairs and sort it: tuples sort by their first item, so the nearest come first. Then count the labels of the first k with a dict (C4's pattern).

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. `neighbours = []`; for each `i`, append `(squared_distance(image, train_images[i]), i)`; then `neighbours.sort()`.
2. Count the labels of `neighbours[:k]` in a dict `votes`, and find the top count with `max(votes.values())`.
3. Walk `neighbours[:k]` again from the nearest and return the first label whose vote equals the top count: that settles a tied vote in favour of the closest neighbour.

</details>

<details>
<summary><strong>Hint 3 · Ask an AI reviewer (keep only what you can explain)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a code reviewer and do NOT rewrite my code. Here is my function and what the check or my test printed: [paste]. Point out one problem at a time and ask me how I would fix it."*

</details>

In [ ]:
# ✏️ WRITE CODE · the experiment: fill in the loop (below the YOUR CODE HERE line), then run (about 15 s).
# Choose k on a validation split of the TRAINING data. To keep it quick, it uses 14 x 14 images:
# downsample() averages each 2 x 2 block (4 times fewer pixels).
scores = {}                                   # k -> accuracy on the validation set
if ready(predict_knn, accuracy, squared_distance):
    small_train = [downsample(image) for image in train_images]
    fit_images, fit_labels, val_images, val_labels = split_train(small_train, train_labels, n_val=200, seed=2026)
    # ===== YOUR CODE HERE =====
    # TODO: for each k in [1, 3, 5, 7]: predict every image in val_images with
    #       predict_knn(image, fit_images, fit_labels, k=k), then put accuracy(guesses, val_labels) in scores[k]
    if scores:
        # ± is the standard error: how much a score on this many images moves by chance alone
        for k in sorted(scores):
            p = scores[k]
            print(f"k = {k}: {p:.1%}  (± {100 * (p * (1 - p) / len(val_images)) ** 0.5:.1f} points)")
        best_k = max(scores, key=scores.get)      # the best validation score (a tie: the smaller k)
        print(f"Chosen on validation: k = {best_k}")
    else:
        print("Write the loop (the TODO in this cell), then run this cell again.")

In [ ]:
# ▶️ JUST RUN · now, and only now, the test set: once, with the k you chose (100 test images, 14 x 14)
if globals().get("scores"):                   # filled by the validation cell above
    small_test = [downsample(image) for image in test_images]
    guesses = [predict_knn(image, small_train, train_labels, k=best_k) for image in small_test]
    print(f"Test accuracy with k = {best_k}: {accuracy(guesses, test_labels):.1%}")
else:
    print("Run the validation cell above first.")

### Explain your result (SCI2)

- Which k won on validation? By how much? Is that gap bigger than the ± next to each score?
- Why is it dishonest to try every k on the test set and report the best one?
- Is a bigger k always better? What does your table say?

### ADVANCED · BLD2 · Robust drawings · Builder track · ✏️ WRITE CODE

Real drawings are off-centre and the wrong size. Write your own version of what prepare_drawing does, and see whether it changes the answer.

**What you'll learn:** cropping, scaling and pasting with 2-D indexing, so that drawn digits of any size and position look like MNIST digits (considerably more work than BLD1) · **Difficulty:** demanding · **Typical time:** 20–25 min

**Signature**

```python
def center_digit(image: list[int]) -> list[int]
```

**Toy example**

```python
# 1. crop to the ink box: the first and last row and column that have a pixel > 0
# 2. scale it so the longer side is 20 pixels (nearest neighbour: each new pixel copies one old pixel)
# 3. paste it in the middle of a new 28 x 28 image of zeros
center_digit(image)    # a list of 784 ints, with the ink centred near (14, 14)
```

The check is a sanity check: 784 ints, the longer side of the ink 20 px, the ink centred within 2 px of (14, 14), and a blank image stays blank.

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).

def center_digit(image: list[int]) -> list[int]:
    """Crop to the ink (pixels > 0), scale the longer side to 20 pixels, and paste it centred in 28 x 28."""
    # TODO 1: the ink box: the first and last row, and the first and last column, with a pixel > 0
    # TODO 2: scale = 20 / the longer side; the new height and width are round(old size * scale)
    # TODO 3: new pixel (r, c) copies old pixel (top + int(r / scale), left + int(c / scale))
    # TODO 4: paste into 784 zeros at row (28 - new height) // 2 and column (28 - new width) // 2
    pass

In [ ]:
# ▶️ JUST RUN · check your work
check("BLD2", center_digit);

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

Three steps: find the ink box; scale it to 20 px by copying, for each new pixel, the old pixel it comes from; paste it into 784 zeros so that it sits in the middle.

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. `rows = [r for r in range(28) if any(image[r * 28 + c] > 0 for c in range(28))]`, and the same for columns. No rows: return `[0] * 784`.
2. `height = rows[-1] - rows[0] + 1` (and `width` alike); `scale = 20 / max(height, width)`; the new size is `round(height * scale)` by `round(width * scale)`.
3. For each new pixel `(r, c)`: it comes from old pixel `(top + int(r / scale), left + int(c / scale))`; write it at `(row_offset + r) * 28 + col_offset + c`, with `row_offset = (28 - new_height) // 2` (and the same for columns).

</details>

<details>
<summary><strong>Hint 3 · Ask an AI reviewer (keep only what you can explain)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a code reviewer and do NOT rewrite my code. Here is my function and what the check or my test printed: [paste]. Point out one problem at a time and ask me how I would fix it."*

</details>

In [ ]:
# ▶️ JUST RUN · before and after, on your drawing from "Draw your own digit" (or the example drawing)
from pathlib import Path
if not str(globals().get("drawing", "")).strip():     # no drawing in memory: use the kit's example
    drawing = Path("tools/drawing_example.txt").read_text(encoding="utf-8")
if ready(center_digit, predict_nearest):
    raw = parse_drawing(drawing)
    mine = center_digit(raw)
    side_by_side([raw, mine], ["as drawn", "after center_digit"])
    print("Read as drawn:", predict_nearest(raw, train_images, train_labels),
          "· after center_digit:", predict_nearest(mine, train_images, train_labels))

### Explain your result (BLD2)

- Did centring change the answer for your drawing? For the example drawing?
- `prepare_drawing` scales smoothly (each new pixel mixes its four nearest old pixels) and centres by the
  *centre of mass* instead of the ink box. Which differences can you see between the two pictures?

### ADVANCED · Challenge: the Pure-Python League · ✏️ WRITE CODE

**What you'll learn:** designing, measuring and improving your own digit reader under a time budget · **Difficulty:** open-ended · **Typical time:** open-ended (you can continue at home)

Build the most accurate digit reader you can in plain Python: it must read all 2,000 public test images within **60 seconds** (normalised for your laptop's speed). `league_run(my_predict, fit_fn=my_fit)` scores it. These are practice rounds: the official ranking is one run on a hidden holdout in Lab 5. AI is your pair programmer here: verify what it suggests, and say how you used it in your Engineering log.

**Ideas to try**

- Measure a first entry before you improve anything: the average digits of SCI1 take two lines.
- Shrink the images to 14 × 14 with `downsample()`: 4 times fewer pixels.
- Several average images per digit: split each digit's images into groups (people write 7s in different ways) and average each group.
- Prune the training set, or stop a distance early once it can't win (ENG2).
- A multiclass **perceptron**: keep one list of weights per digit. To read an image, multiply its pixels by each digit's weights and add them up; the biggest total wins. When the guess is wrong during training, add the image to the right digit's weights and subtract it from the guessed digit's weights. Repeat over the training images a few times.
- For calibration: plain nearest neighbour with 1,000 training images needs about 2 minutes for 2,000 test images, so you'll need trade-offs.

**Rules**

- Standard library only (`math.dist` and `statistics` are allowed).
- Train on the 10,000-image kit subset only. `my_predict` sees one image at a time and never reads test labels; no answers typed in or saved.
- Tune honestly: `league_run(..., on="validation")` learns from 8,000 training images and scores the other 2,000, as often as you like. Run the public test once per idea; that is the number for the board.
- Opt-in, pseudonyms only, nothing collected. The full rules are in `LEAGUE.md`.

**Or choose an alternative**

- **A PNG encoder** (standard library: `zlib` and `struct`, about 20 lines): save digits as real image files. Cells below.
- **The numpy division** (optional, needs numpy and the full 60,000-image MNIST from `tools/get_full_mnist.py`): nearest neighbour over all of it. Rules: no ML frameworks, explain every line, report validation accuracy, never test. Cells below.

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).

# ✏️ WRITE CODE · your League entry: my_fit learns once, my_predict reads one image. Run this cell, then the next one.
from labkit import downsample, league_run

def my_fit(train_images: list[list[int]], train_labels: list[int]) -> object:
    """Learn from the training data once; return your model (any value: a dict, a list, ...)."""
    # A first entry, in one line (SCI1, or from labkit.reference.lab1 import fit_centroids):
    # return fit_centroids(train_images, train_labels)
    pass


def my_predict(image: list[int], model: object) -> int:
    """Read one image with your model; return a label from 0 to 9."""
    # A first entry, in one line: return predict_centroid(image, model)
    pass

In [ ]:
# ▶️ JUST RUN · a practice round: fit once, then predict 2,000 images within the time budget.
# Tune on validation (as often as you like); when an idea is ready, run the public test once:
league_result = league_run(my_predict, fit_fn=my_fit, on="validation")
# league_result = league_run(my_predict, fit_fn=my_fit)          # the public test: the number for the board

#### Alternative: a PNG encoder (ADVANCED, standard library)

A PNG file is an 8-byte signature followed by *chunks*: `IHDR` (width, height, format), `IDAT` (the pixels,
compressed with `zlib`, each row preceded by a `0` byte) and `IEND`. Each chunk is its length (4 bytes),
its type, its data, and a CRC-32 checksum of type + data (`zlib.crc32`). `struct.pack(">I", n)` writes a
number as 4 big-endian bytes. Write paper as white: `255 - pixel`.

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).

import struct
import zlib
from pathlib import Path


def write_png(image: list[int], path: str | Path, width: int = 28) -> None:
    """Save `image` as an 8-bit greyscale PNG file (ink dark on white paper)."""
    # TODO: signature + IHDR (struct.pack(">IIBBBBB", width, height, 8, 0, 0, 0, 0)) + IDAT + IEND
    pass

In [ ]:
# ▶️ JUST RUN · writes test image 0 with your write_png, then reads the file back and checks every pixel
from labkit.lab1_tools import check_png
out = Path("output") / "test_digit_0.png"
out.parent.mkdir(exist_ok=True)
out.unlink(missing_ok=True)        # so that the check below never reads a file from an earlier run
write_png(test_images[0], out)
check_png(out, test_images[0]);    # reads the file back: signature, chunk CRCs, IHDR and every pixel

#### Alternative: the numpy division (ADVANCED: needs numpy and the full MNIST file)

Setup, once: `uv sync --group extras` (or install numpy in your kernel), then in a terminal in the kit
folder: `python tools/get_full_mnist.py` (it saves `data/full/mnist.npz`, about 11 MB). Hold out part of
`x_train` for validation, and **never** use `x_test`: the public test images and the League holdout both
come from it. (After this cell imports numpy, `league_run` counts your entries in the numpy division.)

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).

import importlib.util
import time
from pathlib import Path


def numpy_nearest(npz_path: str | Path = "data/full/mnist.npz", n_val: int = 1000) -> dict:
    """Nearest neighbour over the full training set with numpy, scored on a validation split (never x_test)."""
    # TODO: load x_train and y_train; keep the last 10,000 apart for validation; for n_val of them, find the
    #       nearest of the other 50,000 (|a - b|^2 = |a|^2 - 2 a.b + |b|^2 helps); return accuracy and seconds
    pass

In [ ]:
# ▶️ JUST RUN · runs your numpy_nearest, if numpy and the full MNIST file are there
full = Path("data/full/mnist.npz")
if importlib.util.find_spec("numpy") is None or not full.is_file():
    print("Optional: this needs numpy and data/full/mnist.npz (python tools/get_full_mnist.py). Skipped.")
else:
    result = numpy_nearest(full)
    print(result if result else "Write numpy_nearest first, then run this cell again.")

### Engineering log

Short notes are fine. Edit this cell as you go.

- **What I tried:**
- **What I measured:**
- **What I'd do next:**

## AI-use declaration (about 1 minute)

This course asks you to use AI critically and to say how you used it. This cell stays in your notebook:
nobody collects or grades it. Please don't write your name.

- **Tools I used** (or "none"):
- **For which tasks** (e.g. C2, ENG1):
- **One prompt I used** (copy it, or summarise it in one line):
- **How I checked the answer** (check(), my own test, reading the code, a classmate…):
- **Something the AI got wrong, or that I changed:**
- **One line of AI-written code I can now explain in my own words:**

## Before you go (private, 30 seconds)
- One thing I can do now that I couldn't at the start of this lab:
- One thing I'm still unsure about:

## What's next

- **Catch-up pack**, on IE Campus after class: `lab1_solution.ipynb` (CORE and OPTIONAL solutions), `lab1_express.ipynb` (the minimum path in about 25 minutes, with the same checks) and `lab1_what_you_missed` (a one-page summary).
- **ADVANCED solutions** (including the Challenge) come with the Lab 2 kit, in `solutions_previous/`, so the Challenge stays open until then.
- **Next:** Lab 2 · Read a Postcode, Pythonically, Wed 21 Oct, 15:00–16:20, MMB-703. The kit appears on IE Campus 48 hours before.
- **The League** is opt-in and uses pseudonyms: the rules are in `LEAGUE.md`.
- **Missed something?** `labkit/reference/lab1.py` has a clear, commented version of every CORE and OPTIONAL function of this lab.
- **Your own handwriting:** `tools/draw_digit.html` works offline; try it at home with the canvas route above.